# Statistical hypothesis testing in python
- experiment with stats libraries
- focus on basic foundational stats

## Hypothesis testing process
- a statistical method used to infer population parameters based on sample data
- Two competing hypotheses
- The null hypothesis (H0) which claims there is no effect or difference
- The alternative hypothesis (H1), which claims there is an effect or difference
- Significance Level (α): Define a cutoff point in order to reject the null hypothesis
- Computing a Test Statistic- Form a statistic such as z, t of the sample
- Determine the p-value
- Reject H0 if the p-value ≤ α and fail to reject H0 if it isn’t

In [ ]:
!pip install scipy

In [ ]:
# import the required packages
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns
import scipy.stats as stats

In [ ]:
gene_tx_df = pd.read_csv('genes_transcripts.csv')

In [ ]:
gene_tx_df.head(n=2)

### Is `tx_length` normally distributed?
- many statistical tests assume normality
- test using Shapiro Wilk test
- Hypothesis:
- H0: data is normally distributed
- H1: data is not normally distributed

In [ ]:
from scipy.stats import shapiro
stat, p = shapiro(gene_tx_df["tx_length"])

print(f"Shapiro-Wilk p-value: {p:.4f}")

if p > 0.05:
    print("Data looks approximately normal")
else:
    print("Data is not normally distributed")

In [ ]:
# plot tx_length
gene_tx_df['tx_length'].plot(kind='hist')

#### Because transcript lengths are often highly skewed, normality usually fails.

### Is there a significant difference in `tx_length` between protein_coding and other biotypes?

In [ ]:
# create two groups, one protein_coding, one the rest
protein_coding = gene_tx_df[gene_tx_df['biotype'] == 'protein_coding']['tx_length']
not_protein_coding = gene_tx_df[gene_tx_df['biotype'] != 'protein_coding']['tx_length']

#### Option A — Independent t-test
- The t-statistic compares mean differences relative to variability
- Use when:
- groups are independent
- data roughly normal
- variances similar
- Hypothesis:
- H0: means are equal
- H1: means differ

In [ ]:
from scipy.stats import ttest_ind

stat, p = ttest_ind(protein_coding, not_protein_coding, equal_var=False)

print(f"t-statistic = {stat:.4f}")
print(f"p-value = {p:.4f}")

### Is above one-tailed or two-tailed test?
- can you change to a one-tailed hypothesis
- use alternative='less' or alternative='greater' to indicate if mean of the first group is less than the mean of the second group, or vice-versa resp.

In [ ]:
stat, p = ttest_ind(protein_coding, not_protein_coding, equal_var=False, alternative='')
print(f"t-statistic = {stat:.4f}")
print(f"p-value = {p:.4f}")

#### Option B — Mann–Whitney U Test
- Better when data is skewed.

In [ ]:
from scipy.stats import mannwhitneyu

stat, p = mannwhitneyu(protein_coding, not_protein_coding)

print(f"U statistic = {stat}")
print(f"p-value = {p:.4f}")

### Is there a significant difference in the length of transcripts between BRCA1 and PTEN?


In [ ]:
brca1_tx_len = gene_tx_df[gene_tx_df['gene_name'] == 'BRCA1']['tx_length']
pten_tx_len = gene_tx_df[gene_tx_df['gene_name'] == 'PTEN']['tx_length']
stat, p = mannwhitneyu(brca1_tx_len, pten_tx_len)

print(f"U statistic = {stat}")
print(f"p-value = {p:.4f}")

### Is there a significant difference in transcript length across 6 biotypes?
- comparing means of more than 2 independent groups
#### Option A — One-Way ANOVA
- Hypothesis:
- H0: all group means are equal
- H1: at least one differs

In [ ]:
from scipy.stats import f_oneway

groups = [
    g["tx_length"].values
    for name, g in gene_tx_df.groupby("biotype")
]
print(len(groups))
stat, p = f_oneway(*groups)

print(f"F-statistic = {stat:.4f}")
print(f"p-value = {p:.4f}")

#### Tukey's HSD test

In [ ]:
gene_tx_df.groupby("biotype").size()

In [ ]:
groups = [
    g['tx_length'].values
    for name, g in gene_tx_df.groupby('biotype')
    if len(g['tx_length']) > 1
]

In [ ]:
len(groups)

In [ ]:
from scipy.stats import tukey_hsd

# Perform Tukey's HSD test
res = tukey_hsd(*groups)
print(res)

#### Group 0 has a mean transcript length ~60,609 smaller than Group 1, plausible true difference is between -91,453 and -29,765

### Is there a significant difference in transcript length across 6 biotypes?
- comparing means of more than 2 independent groups
#### Option B — Kruskal Wallis
- The Kruskal-Wallis test is primarily used when  data violates strict assumptions like normality or equal variances, relying instead on comparing group medians
- Hypothesis:
- H0: all group means are equal
- H1: at least one differs

In [ ]:
from scipy.stats import kruskal
groups = [
    g["tx_length"].values
    for name, g in gene_tx_df.groupby("biotype")
]
stat, p = kruskal(*groups)

print(f"H-statistic = {stat:.4f}")
print(f"p-value = {p:.4f}")

### Chi-Square test of independence
- Is biotype associated with strand direction?
- Hypothesis:
- H0: variables are independent
- H1: variables are associated

In [ ]:
# create a freq table showing counts of occurrences for combinations of variables
table = pd.crosstab(gene_tx_df["biotype"], gene_tx_df["strand"])

print(table)

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(table)

print(f"chi2 = {chi2:.4f}")
print(f"p-value = {p:.4f}")

### KS test
Useful for comparing two sample distributions
- Hypotheses:
- H0: both samples come from same distribution
- H1: distributions differ
- Unlike t-tests, KS detects differences in shape, spread, skewness and median not just mean differences.

#### Do the distributions of tx_length differ for nonsense_mediated_decay and retained_intron biotypes?

In [ ]:
groups = [
    g['tx_length'].values
    for name, g in gene_tx_df.groupby('biotype')
    if len(g['tx_length']) > 1
]

In [ ]:
[
    name
    for name, g in gene_tx_df.groupby('biotype')
    if len(g['tx_length']) > 1
]

In [ ]:
from scipy.stats import ks_2samp

stat, p = ks_2samp(groups[1], groups[5])
print(f"stat = {stat:.4f}")
print(f"p-value = {p:.4f}")

In [ ]:
groups[1].mean(), groups[5].mean()

### Pearson correlation test
- H0: no linear correlation
- H1: linear correlation

#### Are the start and end correlated for transcripts ?

In [ ]:
result = stats.pearsonr(gene_tx_df['start'], gene_tx_df['end'])
print(f"Correlation coefficient (r): {result.statistic:.4f}")
print(f"P-value: {result.pvalue:.4e}")

### Linear regression models

In [ ]:
# plot data and a linear regression model fit
mpg = sns.load_dataset("mpg")
sns.regplot(data=mpg, x="weight", y="acceleration")


### add annotations

In [ ]:
# add annotations
from scipy.stats import linregress


# Linear regression
result = linregress(mpg["weight"], mpg["acceleration"])

# Extract statistics
slope = result.slope
intercept = result.intercept
r_value = result.rvalue
r_squared = r_value**2
p_value = result.pvalue
std_err = result.stderr

# Plot
plt.figure(figsize=(8,6))

sns.regplot(
    data=mpg,
    x="weight",
    y="acceleration",
    line_kws={"color": "red"}
)

# Annotation text
text = (
    f"Slope = {slope:.4f}\n"
    f"Intercept = {intercept:.4f}\n"
    f"R² = {r_squared:.4f}\n"
    f"p-value = {p_value:.4e}\n"
    f"Std Err = {std_err:.4f}"
)

# Put text on plot
plt.text(
   4000,
    25,
    text,
    fontsize=10,
    verticalalignment='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

plt.title("Weight vs Acceleration")
plt.show()

### Create separate linear regressions for multiple variables using the penguins dataset
- plot bill length versus bill depth for various species of penguins

In [ ]:
penguins = sns.load_dataset("penguins")
g = sns.lmplot(
    data=penguins,
    x="bill_length_mm", y="bill_depth_mm", hue="species",
    height=5
)
g.set_axis_labels("bill length (mm)", "bill depth (mm)")

### Multiple linear regression
- uses multiple predictors to predict one outcome
- `bill_depth_mm ~ bill_length_mm + body_mass_g + flipper_length_mm`

- So the fitted model is:

- `bill_depth=β0 +β1(bill_length)+β2(body_mass)+β3(flipper_length)`

In [ ]:
!pip install statsmodels

In [ ]:
import statsmodels.formula.api as smf

penguins = sns.load_dataset("penguins").dropna()

model = smf.ols(
    "bill_depth_mm ~ bill_length_mm + body_mass_g + flipper_length_mm",
    data=penguins
).fit()

print(model.summary())

#### The model explains 37.6% of the variation in bill depth
- p-value = 2.03e-33
- H0: all coefficients = 0 (no predictive power)
- H1: at least one predictor matters
- Since p is extremely small the model is statistically significant overall.

# Collaborative exercise

## Exercise 1
- Using the `genes_transcripts.csv` dataset can you figure out a multiple linear regression problem? Formulate the question, fit the model and interpret the results

### Overall independent project guidance
- docstrings: https://peps.python.org/pep-0257/
- PEP8 guidelines: https://peps.python.org/pep-0008/
- working with files, see https://python-adv-web-apps.readthedocs.io/en/latest/working_with_files.html
